In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch9. RAG</font>

# 문장 → 임베딩 벡터(1차원 숫자 배열)
- OpenAI의 text-embedding-3-large(text-embedding-3-small) 모델 활용
    - .env 파일에 OPENAI_API_KEY 추가
- UpStage의 embedding-query 모델 활용
    - .env 파일에 UPSTAGE_API_KEY 추가

# 1. 환경 변수 load

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')
upstage_key = os.getenv('UPSTAGE_API_KEY')

# print(f"OPENAI_API_KEY: {openai_key}")
# print(f"UPSTAGE_API_KEY: {upstage_key}")

# 2. 유사도 계산
1. 유클리드 계산법 : 두 벡터 사이의 최단 직선 거리를 계산
2. 코사인 유사도 계산법 : 두 벡터가 이루는 각도(방향)를 기준으로 유사도를 계산
3. dot product 유사도 계산법 : 두 벡터의 성분별 곱의 합으로, 벡터의 거리(크기)와 방향을 모두 복합적으로 고려하여 유사도를 계산

In [8]:
import numpy as np
def cosine_similarity(v1, v2):
    '두 벡터의 코사인 유사도를 반환'
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0
    return dot_product / (norm_v1 * norm_v2)

In [5]:
v1 = np.array([3, 4])
v2 = np.array([3, 4.1])
v3 = np.array([25, -9])

In [11]:
print(cosine_similarity(v1, v2))
print(cosine_similarity(v2, v3))

0.9999302571572745
0.2822465375117816


# 3. OpenAI의 embeddings model 사용
- text-embedding-3-large(text-embedding-3-small)

In [13]:
from openai import OpenAI
openai_client = OpenAI()

In [47]:
import openai
openai.__version__

'3.11.0'

In [ ]:
# text-embedding-3-small : 1536차원(1536열)의 벡터로 변환
response = openai_client.embeddings.create(
    input = "The king is the prince's father",
    model = "text-embedding-3-small"
    )

In [17]:
print(response.usage)
print(len(response.data[0].embedding))

Usage(prompt_tokens=7, total_tokens=7)
1536


In [19]:
king_vector = np.array(response.data[0].embedding)
print(king_vector.shape)

(1536,)


In [25]:
# text-embedding-3-large : 3072차원(3072열)의 벡터로 변환
response1 = openai_client.embeddings.create(
    input = "The queen is the prince's mother",
    model = "text-embedding-3-large"
    )

In [26]:
# text-embedding-3-large : 3072차원(3072열)의 벡터로 변환
response2 = openai_client.embeddings.create(
    input = "The king is the prince's father",
    model = "text-embedding-3-large"
    )

In [27]:
print(response.usage)
print(len(response.data[0].embedding))

Usage(prompt_tokens=7, total_tokens=7)
3072


In [28]:
queen_vector = np.array(response1.data[0].embedding)
king_vector = np.array(response2.data[0].embedding)

In [29]:
# king 문장과 queen 문장의 유사도 확인
cosine_similarity(king_vector, queen_vector)

np.float64(0.6953830686006353)

In [32]:
response = openai_client.embeddings.create(
    input = "The Slave is begged",
    model = "text-embedding-3-large"
    )

In [33]:
slave_vector = np.array(response.data[0].embedding)

In [ ]:
# king 문장과 slave 문장의 유사도 확인
cosine_similarity(king_vector, slave_vector)

np.float64(0.1780269722087026)

In [35]:
response = openai_client.embeddings.create(
    input = "왕은 왕자의 아버지다",
    model = "text-embedding-3-large"
    )

In [36]:
kor_king_vector = np.array(response.data[0].embedding)

In [37]:
response = openai_client.embeddings.create(
    input = "여왕은 왕자의 어머니다",
    model = "text-embedding-3-large"
    )

In [38]:
kor_queen_vector = np.array(response.data[0].embedding)

In [39]:
response = openai_client.embeddings.create(
    input = "노예가 구걸한다",
    model = "text-embedding-3-large"
    )

In [40]:
kor_slave_vector = np.array(response.data[0].embedding)

In [45]:
print(cosine_similarity(kor_queen_vector, kor_slave_vector))
print(cosine_similarity(kor_queen_vector, kor_king_vector))
print(cosine_similarity(kor_slave_vector, kor_king_vector))

0.19003438359918243
0.576352799705736
0.1356222913223156


# 4. upstage의 embedding model 사용
- 한국어 환경에서의 성능 우수

In [50]:
upstage_client = OpenAI(
    api_key = upstage_key,
    base_url="https://api.upstage.ai/v1"
    )

In [56]:
response = upstage_client.embeddings.create(
    input = "The king is prince's father",
    model="embedding-query"
)

In [57]:
up_king_vector = np.array(response.data[0].embedding)
print(up_king_vector.shape)

(4096,)


In [59]:
response = upstage_client.embeddings.create(
    input = "The queen is prince's mother",
    model="embedding-query"
)

up_queen_vector = np.array(response.data[0].embedding)

In [61]:
response = upstage_client.embeddings.create(
    input = "The Slave begs",
    model="embedding-query"
    )

up_slave_vector = np.array(response.data[0].embedding)

In [62]:
response = upstage_client.embeddings.create(
    input = "왕은 왕자의 아버지다",
    model="embedding-query"
)

up_kor_king_vector = np.array(response.data[0].embedding)

In [63]:
response = upstage_client.embeddings.create(
    input = "여왕은 왕자의 어머니다",
    model="embedding-query"
)

up_kor_queen_vector = np.array(response.data[0].embedding)

In [64]:
response = upstage_client.embeddings.create(
    input = "노예가 구걸한다",
    model="embedding-query"
)

up_kor_slave_vector = np.array(response.data[0].embedding)

In [66]:
print(cosine_similarity(up_king_vector, up_queen_vector))
print(cosine_similarity(up_slave_vector, up_queen_vector))
print(cosine_similarity(up_king_vector, up_slave_vector))

0.6697470062132704
0.32301138053005773
0.3126250552634183


In [67]:
print(cosine_similarity(up_kor_king_vector, up_kor_queen_vector))
print(cosine_similarity(up_kor_slave_vector, up_kor_queen_vector))
print(cosine_similarity(up_kor_king_vector, up_kor_slave_vector))

0.650055435904611
0.31776173293536875
0.33553422773239017


In [69]:
print(cosine_similarity(up_king_vector, up_kor_king_vector))
print(cosine_similarity(up_queen_vector, up_kor_queen_vector))

0.8492781620590963
0.802296850297972
